In [1]:
!pip install ultralytics roboflow -q

import ultralytics
ultralytics.checks()

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.1/112.6 GB disk)


In [2]:
from roboflow import Roboflow

rf = Roboflow(api_key="PNdPWEJrlkZK96y27Fiy")
project = rf.workspace("sovithyeas-workspace").project("formula-1-sexm0-yhlv0")
version = project.version(1)
dataset = version.download("yolov8")

DATASET_PATH = dataset.location
print(f"Dataset downloaded to: {DATASET_PATH}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Formula-1--1 in yolov8:: 100%|██████████| 965/965 [00:00<00:00, 4478.68it/s]

Dataset downloaded to: /content/Formula-1--1


In [3]:
import os

for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for f in files[:3]:
            print(f'{subindent}{f}')
        if len(files) > 3:
            print(f'{subindent}... ({len(files)} files total)')

Formula-1--1/
  README.dataset.txt
  data.yaml
  README.roboflow.txt
  train/
    images/
    labels/
  valid/
    images/
    labels/
  test/
    images/
    labels/


In [4]:
import yaml

yaml_path = os.path.join(DATASET_PATH, "data.yaml")
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print(yaml.dump(data_config, default_flow_style=False))
print(f"Number of classes: {data_config['nc']}")
print(f"Class names: {data_config['names']}")

names:
- Alpine
- AstonMartin
- Ferrari
- Mclaren
- Mercedes
- RedBull
- Williams
- f1car - v3 2024-05-13 1-30pm
nc: 8
roboflow:
  license: CC BY 4.0
  project: formula-1-sexm0-yhlv0
  url: https://universe.roboflow.com/sovithyeas-workspace/formula-1-sexm0-yhlv0/dataset/1
  version: 1
  workspace: sovithyeas-workspace
test: ../test/images
train: ../train/images
val: ../valid/images

Number of classes: 8
Class names: ['Alpine', 'AstonMartin', 'Ferrari', 'Mclaren', 'Mercedes', 'RedBull', 'Williams', 'f1car - v3 2024-05-13 1-30pm']


In [5]:
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATASET_PATH, split, 'images')
    if os.path.exists(img_dir):
        count = len([f for f in os.listdir(img_dir)
                     if f.endswith(('.jpg', '.jpeg', '.png'))])
        print(f"{split}: {count} images")

train: 423 images
valid: 31 images
test: 26 images


In [6]:
label_dir = os.path.join(DATASET_PATH, 'train', 'labels')
sample_label = os.listdir(label_dir)[0]

print(f"Label file: {sample_label}")
print("Contents:")
with open(os.path.join(label_dir, sample_label), 'r') as f:
    print(f.read())

Label file: 00000532_jpg.rf.46627029bdeeb7dfaf0e44984838fb1e.txt
Contents:
2 0.48515625 0.53671875 0.8375 0.60625


In [7]:
from ultralytics import YOLO

pretrained_model = YOLO('yolov8n.pt')

In [8]:
pretrained_results = pretrained_model.val(
    data=yaml_path,
    split='test',
    imgsz=640,
    verbose=True
)

# Extract the metrics
print("\nPretrained Model Results")
print(f"mAP50: {pretrained_results.box.map50:.4f}")
print(f"mAP50-95: {pretrained_results.box.map:.4f}")
print(f"Precision: {pretrained_results.box.mp:.4f}")
print(f"Recall: {pretrained_results.box.mr:.4f}")

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1157.5±231.7 MB/s, size: 59.5 KB)
val: Scanning /content/Formula-1--1/test/labels... 26 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 26/26 508.2it/s 0.1s
val: New cache created: /content/Formula-1--1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.1it/s 1.9s
                   all         26         32      0.408      0.283      0.202      0.164
                person          2          2          0          0          0          0
               bicycle          1          1          1          0     0.0663     0.0597
                   car          4          5      0.144        0.2      0.127      0.116
            motorcycle          6          8      0.162      0.362      0.185  

In [9]:
model = YOLO('yolov8n.pt')

train_results = model.train(
    data=yaml_path,
    epochs=30,
    imgsz=640,
    freeze=10,
    batch=16,
    patience=10,
    project='yolo_lab6',
    name='finetune',
    verbose=True
)

print("Training complete!")

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Formula-1--1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=finetune, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=1

In [11]:
finetuned_model = YOLO('/content/runs/detect/yolo_lab6/finetune/weights/best.pt')

finetuned_results = finetuned_model.val(
    data=yaml_path,
    split='test',
    imgsz=640,
    verbose=True
)

print("\nFine-tuned Model Results")
print(f"mAP50: {finetuned_results.box.map50:.4f}")
print(f"mAP50-95: {finetuned_results.box.map:.4f}")
print(f"Precision: {finetuned_results.box.mp:.4f}")
print(f"Recall: {finetuned_results.box.mr:.4f}")

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1592.3±621.5 MB/s, size: 52.3 KB)
val: Scanning /content/Formula-1--1/test/labels.cache... 26 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 26/26 7.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.4it/s 1.4s
                   all         26         32      0.708      0.556      0.628      0.532
                Alpine          2          2      0.421        0.5      0.695      0.635
           AstonMartin          1          1          1          0          0          0
               Ferrari          4          5          1      0.964      0.995      0.792
               Mclaren          6          8      0.801       0.75       0.91      0.737
              Mercedes          4          4       

In [14]:

print("PRETRAINED vs FINE-TUNED COMPARISON")
print("-" * 45)
print(f"{'Metric':<15} {'Pretrained':>12} {'Fine-tuned':>12}")
print("-" * 45)
print(f"{'mAP50':<15} {0.2019:>12.4f} {0.6281:>12.4f}")
print(f"{'mAP50-95':<15} {0.1637:>12.4f} {0.5323:>12.4f}")
print(f"{'Precision':<15} {0.4080:>12.4f} {0.7085:>12.4f}")
print(f"{'Recall':<15} {0.2828:>12.4f} {0.5559:>12.4f}")
print("-" * 45)

PRETRAINED vs FINE-TUNED COMPARISON
---------------------------------------------
Metric            Pretrained   Fine-tuned
---------------------------------------------
mAP50                 0.2019       0.6281
mAP50-95              0.1637       0.5323
Precision             0.4080       0.7085
Recall                0.2828       0.5559
---------------------------------------------
